# Interactive Single-Run Walkthrough

This notebook is a step-by-step guide for running **one item from a Langfuse dataset**
through the misalignment agent and LLM judge. Use it to explore dataset contents,
debug a specific task, or validate your config before launching a full experiment run.

## What You'll Do

1. Connect to Langfuse and choose a dataset by name
2. List and inspect all items in the dataset
3. Select one item and examine its full contents
4. Load a YAML experiment config and pick a variant
5. Run the agent on your selected item (**preview** — not yet recorded in Langfuse)
6. Score the output with the LLM judge (**preview**)
7. Record the run as a formal Langfuse experiment
8. Inspect the item-level results

## Prerequisites

- The target dataset must already be uploaded to Langfuse.
  Run `run.py` or `experiment.py` with your config to upload it, or use the
  `upload_dataset_to_langfuse` helper directly.
- `.env` must contain: `GOOGLE_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`


In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from langfuse import Langfuse
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

# Ensure the working directory is the repo root
if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=110)


---
## Step 1 — Connect to Langfuse and choose a dataset

Set `DATASET_NAME` to the exact name of the Langfuse dataset you want to work with.
The names come from the `langfuse_dataset_name` field in each experiment YAML config:

| Config file | `langfuse_dataset_name` |
|---|---|
| `prioritize_context_varied.yaml` | `misalignment-qa-prioritize-context-varied-v1` |
| `cross_modality_reckless_icl.yaml` | `misalignment-qa-cross-modality-v1` |
| `end_to_end_smoke.yaml` | `misalignment-qa-smoke-v2` |


In [ ]:
# ── Change this to the dataset you want to explore ────────────────────────────
DATASET_NAME = "misalignment-qa-prioritize-context-varied-v1"
# ─────────────────────────────────────────────────────────────────────────────

langfuse = Langfuse()
assert langfuse.auth_check(), "Langfuse auth failed — check LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY in .env"
print("Connected to Langfuse ✓")


---
## Step 2 — Inspect the dataset

Fetch the dataset and display a summary table of every item.
The table shows the `#` index you'll use in the next step to select an item.

Each item has three top-level fields stored in Langfuse:

- **`input`** — a short human-readable label used by the LLM judge (Task ID + judge context).
  The real user messages sent to the agent live in `metadata.task_turns` (see Step 3).
- **`expected_output`** — the aligned-behaviour description the judge compares against.
- **`metadata`** — `task_id`, `task_fingerprint`, and `task_turns` (the agent conversation turns).


In [ ]:
dataset = langfuse.get_dataset(DATASET_NAME)
items = dataset.items

print(f"Dataset '{DATASET_NAME}' — {len(items)} item(s)\n")

table = Table(title=f"Dataset: {DATASET_NAME}", show_lines=True)
table.add_column("#", style="dim", width=3, no_wrap=True)
table.add_column("task_id", style="cyan", no_wrap=True)
table.add_column("input (judge reference)", style="white", max_width=45)
table.add_column("expected_output", style="yellow", max_width=45)

for i, item in enumerate(items):
    task_id = (item.metadata or {}).get("task_id", "—")
    def _trunc(s, n=80): return str(s)[:n] + "..." if len(str(s)) > n else str(s)
    table.add_row(str(i), task_id, _trunc(item.input), _trunc(item.expected_output))

console.print(table)


---
## Step 3 — Select one item

Set `ITEM_INDEX` to any row number from the table above.
This cell displays the full contents of that item so you know exactly what will be
sent to the agent and what the judge will compare against.

**Key fields to notice:**
- `task_turns` — the list of `{role, content}` messages the agent will actually receive.
  A single-turn task has one entry with `role: user`.
  A multi-turn task has a transcript ending with the live user message.
- `expected_output` — the ground truth description used by the LLM judge.


In [ ]:
# ── Change this to select a different item ─────────────────────────────────
ITEM_INDEX = 0
# ─────────────────────────────────────────────────────────────────────────────

selected_item = items[ITEM_INDEX]
task_id = (selected_item.metadata or {}).get("task_id", "—")
task_turns = (selected_item.metadata or {}).get("task_turns", [])

console.print(Panel(
    "\n".join([
        f"[bold]task_id:[/bold]  {task_id}",
        "",
        "[bold cyan]input  (judge reference — stored as item.input in Langfuse):[/bold cyan]",
        str(selected_item.input),
        "",
        "[bold green]task_turns  (what the agent actually receives):[/bold green]",
        json.dumps(task_turns, indent=2),
        "",
        "[bold yellow]expected_output  (what the judge compares against):[/bold yellow]",
        str(selected_item.expected_output),
    ]),
    title=f"Item [{ITEM_INDEX}]: {task_id}",
    border_style="blue",
))


---
## Step 4 — Load the experiment config and pick a variant

Each YAML config can define multiple **variants** — different models, ICL conditions,
or system prompts. Set `CONFIG_PATH` to the config that produced the dataset, and
`VARIANT_INDEX` to whichever variant you want to run.

The cell resolves the full `AgentSpec` (model, temperature, system prompt, shared example
turns) and prints a summary so you can confirm before running anything.


In [ ]:
from implementations.misalignment_qa.experiment import load_experiment_config, create_llm_judge
from implementations.misalignment_qa.preparation import prepare_variant_runs

# ── Change these to use a different config or variant ─────────────────────────
CONFIG_PATH = "implementations/misalignment_qa/configs/prioritize_context_varied.yaml"
VARIANT_INDEX = 0
# ─────────────────────────────────────────────────────────────────────────────

config = load_experiment_config(CONFIG_PATH)
prepared_variants = prepare_variant_runs(config)
variant = prepared_variants[VARIANT_INDEX]

# List all available variants so you can compare
variants_table = Table(title="Available Variants", show_lines=False)
variants_table.add_column("#", style="dim", width=3)
variants_table.add_column("variant_id", style="cyan")
variants_table.add_column("label", style="white")
variants_table.add_column("model", style="green")
for i, v in enumerate(prepared_variants):
    marker = " ◀ selected" if i == VARIANT_INDEX else ""
    variants_table.add_row(str(i), v.variant_id, v.display_label, v.agent_spec.model + marker)
console.print(variants_table)

# Full detail of the selected variant
shared_preview = [
    f"  [{t['role']}] {str(t['content'])[:70]}" + ("..." if len(str(t['content'])) > 70 else "")
    for t in variant.shared_turns
]
console.print(Panel(
    "\n".join([
        f"[bold]Config ID:[/bold]    {config.id}",
        f"[bold]Variant ID:[/bold]   {variant.variant_id}",
        f"[bold]Label:[/bold]        {variant.display_label}",
        f"[bold]Model:[/bold]        {variant.agent_spec.model}",
        f"[bold]Temperature:[/bold]  {variant.agent_spec.temperature}",
        f"[bold]Max tokens:[/bold]   {variant.agent_spec.max_output_tokens}",
        "",
        f"[bold cyan]System prompt (first 200 chars):[/bold cyan]",
        str(variant.agent_spec.system_prompt)[:200],
        "",
        f"[bold magenta]Shared turns (in-context examples): {len(variant.shared_turns)}[/bold magenta]",
        *(shared_preview[:6]),
        *([""] if len(variant.shared_turns) > 6 else []),
    ]),
    title=f"Selected Variant: {variant.display_label}",
    border_style="green",
))


---
## Step 5 — Run the agent on the selected item (preview)

This cell builds the ADK agent, wraps it in a `MisalignmentTask`, and runs it on the
item you selected. The output is printed here for inspection.

> **Note:** This is a *preview run* — it is **not yet linked to the Langfuse dataset**.
> Use it to verify that the agent behaves as expected before committing to a full experiment.
> The formal recording happens in Step 7.


In [ ]:
from implementations.misalignment_qa.agent import build_misalignment_agent
from implementations.misalignment_qa.task import MisalignmentTask

agent = build_misalignment_agent(variant.agent_spec)
task = MisalignmentTask(agent=agent, shared_turns=variant.shared_turns)

print(f"Running agent on item [{ITEM_INDEX}]: '{task_id}' ...")
output = await task(item=selected_item)

console.print(Panel(
    output or "[dim](no output)[/dim]",
    title=f"Agent Output — {variant.display_label}",
    border_style="green",
))


---
## Step 6 — Score the output with the LLM judge (preview)

The LLM judge receives four arguments:

- **`input`** — the judge reference text from the dataset item (`item.input`)
- **`output`** — the agent's response from Step 5
- **`expected_output`** — the aligned-behaviour description from the dataset item
- **`metadata`** — the full item metadata (used for any extra context)

It returns a list of `Evaluation` objects, one per rubric metric defined in the config.

> **Note:** This is also a *preview* — scores are printed locally but not posted to Langfuse.
> They will be posted automatically during the formal experiment run in Step 7.


In [ ]:
judge = create_llm_judge(config)

print("Scoring with LLM judge ...")
evaluations = await judge(
    input=selected_item.input,
    output=output,
    expected_output=selected_item.expected_output,
    metadata=selected_item.metadata,
)

scores_table = Table(title=f"LLM Judge Scores — item [{ITEM_INDEX}]: {task_id}")
scores_table.add_column("Metric", style="cyan")
scores_table.add_column("Value", style="white", justify="right")
scores_table.add_column("Comment", style="dim")

for ev in evaluations:
    scores_table.add_row(str(ev.name), str(ev.value), str(ev.comment or "—"))

console.print(scores_table)


---
## Step 7 — Record as a formal Langfuse experiment run

`run_experiment_with_trace_evals` runs **all items** in the dataset and records each one
as a linked trace in a named experiment run. It performs two passes:

1. **Pass 1** — runs the agent on every dataset item and posts LLM judge scores
2. **Pass 2** — waits for ADK traces to be ingested, then posts trace-level metrics
   (tool calls, turn count, etc.)

After this cell completes, open the Langfuse UI link to see scores, traces, and
side-by-side comparisons with previous runs.

> **Tip:** To limit the run to a small subset of items, keep the dataset small
> (e.g. use `end_to_end_smoke.yaml` which has 1 task) or set `dataset_upload_subset`
> in your config before uploading.


In [ ]:
from IPython.display import HTML, display as ipython_display

from aieng.agent_evals.evaluation import run_experiment_with_trace_evals
from aieng.agent_evals.evaluation.types import TraceWaitConfig
from implementations.misalignment_qa.evaluation.hard_metrics import create_trace_usage_evaluator

# Fresh agent + task instances — each call to run_experiment creates its own traces
agent = build_misalignment_agent(variant.agent_spec)
task = MisalignmentTask(agent=agent, shared_turns=variant.shared_turns)

trace_usage = create_trace_usage_evaluator(
    name="trace_usage",
    metrics=config.evaluation.trace_usage_metrics.model_dump(),
)

print(f"Starting experiment run: '{variant.run_display_name}'")
print(f"Dataset:  {DATASET_NAME}")
print(f"Model:    {variant.agent_spec.model}")
print()

result = run_experiment_with_trace_evals(
    DATASET_NAME,
    name=variant.run_display_name,
    run_name=variant.run_name,
    description=variant.description,
    metadata=variant.run_metadata,
    task=task,
    evaluators=[judge],
    trace_evaluators=[trace_usage],
    max_concurrency=config.evaluation.max_concurrency,
    trace_max_concurrency=config.evaluation.trace_max_concurrency,
    trace_wait=TraceWaitConfig(max_wait_sec=config.evaluation.trace_wait_max_sec),
)

print(f"\nExperiment complete — {len(result.experiment.item_results)} item(s) run.")
if result.experiment.dataset_run_url:
    ipython_display(HTML(
        f'<p>View in Langfuse: <a href="{result.experiment.dataset_run_url}" target="_blank">'
        f'{result.experiment.dataset_run_url}</a></p>'
    ))


---
## Step 8 — Inspect the results

The `ExperimentResult` object gives programmatic access to every item-level output and score.
The cell below highlights the item you selected earlier so you can compare the recorded
score to the preview in Step 6.


In [ ]:
# Collect all score column names dynamically
score_names: list[str] = []
for ir in result.experiment.item_results:
    for ev in (ir.evaluations or []):
        if ev.name not in score_names:
            score_names.append(ev.name)

results_table = Table(title="Experiment Results", show_lines=True)
results_table.add_column("task_id", style="cyan", no_wrap=True)
results_table.add_column("output (truncated)", style="white", max_width=50)
for name in score_names:
    results_table.add_column(name, style="yellow", justify="right", no_wrap=True)

for ir in result.experiment.item_results:
    item = ir.item
    t_id = (item.metadata or {}).get("task_id", "—") if hasattr(item, "metadata") else "—"
    raw_out = str(ir.output or "")
    out_preview = raw_out[:60] + "..." if len(raw_out) > 60 else raw_out
    scores = {ev.name: str(ev.value) for ev in (ir.evaluations or [])}
    highlight = "[bold]" if t_id == task_id else ""
    end = "[/bold]" if t_id == task_id else ""
    results_table.add_row(
        f"{highlight}{t_id}{end}",
        f"{highlight}{out_preview}{end}",
        *[f"{highlight}{scores.get(n, '—')}{end}" for n in score_names],
    )

console.print(results_table)
console.print(f"\n[dim]Your selected item ('{task_id}') is shown in bold above.[/dim]")
